<!-- FULL: keep, DEMO: keep -->
<div style='display:flex; align-items:center; justify-content:space-between; border-bottom:3px solid rgb(255,106,0); padding-bottom:1em;'>
<div>
<span style='color:rgb(22,60,105); font-size:1.8em; font-weight:bold;'>Introduction to Deep Learning</span><br>
<span style='color:rgb(0,85,100); font-size:1.3em;'>Session 6 &mdash; 4: Data Pipeline &mdash; Dataset &amp; DataLoader</span><br>
<span style='color:rgb(0,85,100); font-size:1.0em;'>Magda Gregorová &nbsp;·&nbsp; THWS &nbsp;·&nbsp; May 2026</span>
</div>
<img src='../../Common/Pics/thws-logo_vert_en_orange-rgb.png' style='height:80px;'/>
</div>

<!-- FULL: keep, DEMO: delete -->
*We can build networks. Next: load data efficiently with Dataset and DataLoader.*

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
import matplotlib.pyplot as plt
import sys
sys.path.append('../../A2/')
from helpers import load_data

X, y = load_data('../../A2/data.csv')


<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Custom <code>Dataset</code>
</div>

<!-- FULL: keep, DEMO: delete -->
Subclass `torch.utils.data.Dataset` and implement:
- `__len__`: number of examples
- `__getitem__`: return the $i$-th example

Works for data too large to fit in memory, data loaded from disk, or on-the-fly augmentation.

In [ ]:
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split

class HousingDataset(Dataset):
    """Housing price dataset."""

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


dataset = HousingDataset(X, y)
print(f'Dataset size: {len(dataset)}')
x0, y0 = dataset[0]
print(f'First example: x={x0}, y={y0.item():.4f}')

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  <code>TensorDataset</code> shortcut
</div>

<!-- FULL: keep, DEMO: delete -->
When data fits in memory as tensors, `TensorDataset` wraps them without subclassing.

In [ ]:
dataset = TensorDataset(X, y)
print(f'TensorDataset size: {len(dataset)}')
x0, y0 = dataset[0]
print(f'First example: x shape={x0.shape}, y={y0.item():.4f}')

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Train / validation split
</div>

<!-- FULL: keep, DEMO: delete -->
`random_split` divides a dataset into non-overlapping subsets reproducibly.

In [ ]:
torch.manual_seed(0)
n_train = int(0.8 * len(dataset))
n_val   = len(dataset) - n_train
train_dataset, val_dataset = random_split(dataset, [n_train, n_val])

print(f'Train size:      {len(train_dataset)}')
print(f'Validation size: {len(val_dataset)}')

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  <code>DataLoader</code> — batching and shuffling
</div>

<!-- FULL: keep, DEMO: delete -->
`DataLoader` wraps a `Dataset` and handles mini-batching, shuffling, and parallel loading.

Always `shuffle=True` for training, `shuffle=False` for validation.

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)

print(f'Batches per epoch (train): {len(train_loader)}')
print(f'Batches per epoch (val):   {len(val_loader)}')

# inspect one batch
X_batch, y_batch = next(iter(train_loader))
print(f'\nBatch shapes: X={X_batch.shape}, y={y_batch.shape}')

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(0,85,100); background:rgb(235,245,247); color:rgb(0,85,100); font-size:1.1em; font-weight:bold;'>
  &#8644; Alternatives &mdash; train / validation split approaches
</div>

<!-- FULL: keep, DEMO: delete -->
- `random_split` — simple, built-in, reproducible with `torch.manual_seed`
- Manual index slicing — full control over which examples go where
- `sklearn.model_selection.train_test_split` — more options (stratified, grouped)

In [ ]:
# Option 1 — random_split (already shown)
train_ds, val_ds = random_split(dataset, [0.8, 0.2],
                                generator=torch.Generator().manual_seed(0))

# Option 2 — manual index split
n       = len(dataset)
indices = torch.randperm(n, generator=torch.Generator().manual_seed(0))
n_train = int(0.8 * n)
train_ds2 = torch.utils.data.Subset(dataset, indices[:n_train])
val_ds2   = torch.utils.data.Subset(dataset, indices[n_train:])

print(f'random_split:  train={len(train_ds)}, val={len(val_ds)}')
print(f'manual split:  train={len(train_ds2)}, val={len(val_ds2)}')

<!-- FULL: keep, DEMO: keep -->
<br/>